# Lucy tutorial (Jupyter)

Portable live-fit measuring — **why**: one Go core, no formula forks.
**What**: feed finished cells → LPD board → CSV / site PDF / charts.

Run with repo checkout; binary at `python/src/lucy/bin/lucy`.


In [ ]:
import json, sys
from pathlib import Path
ROOT = Path.cwd()
# allow running from examples/python or repo root
for cand in [ROOT, ROOT.parent.parent, ROOT.parent]:
    if (cand / "python" / "src" / "lucy").is_dir():
        ROOT = cand
        break
sys.path.insert(0, str(ROOT / "python" / "src"))
from lucy import build_lpd, floors, board_records, board_csv, write_site_pdf, __version__, version
print("lucy", __version__, "bin", version())
print("floors", floors()["keep_floor"])


In [ ]:
req = json.loads((ROOT / "examples/shared/samples.json").read_text())
resp = build_lpd(req["samples"], req.get("options"))
board = resp["board"]
for i, r in enumerate(board["top"][:4]):
    print(f"#{i+1} {r['id']:6} band={r['band']:4} LPD={r['lpd']:.4g}")
assert board["top"][0]["id"] == "int8"
print("OK — int8 leads (gold); bin is trap at LPD 0")


In [ ]:
# Optional pandas table
rows = board_records(resp)
try:
    from lucy import to_dataframe
    display(to_dataframe(resp))
except Exception as e:
    print("records", len(rows), "(install pandas for DataFrame)", e)


In [ ]:
out = ROOT / "examples" / "out" / "jupyter"
out.mkdir(parents=True, exist_ok=True)
(out / "board.csv").write_text(board_csv(req["samples"], req.get("options")))
write_site_pdf(req["samples"], out / "site.pdf", req.get("options"))
print("wrote", out / "board.csv", "and", out / "site.pdf")
